# Paper §6.3 — Capability preservation (6-benchmark VLMEvalKit)

Reproduces the §6.3 STRICT_FREE_LUNCH headline: at the same chosen cell `L=26 K=8 α=1.0`, the mitigation applied at inference does not degrade general capability across 6 held-out VLMEvalKit benchmarks (n_total ≈ 10,507). Macro Δ = +0.41 pp; HallusionBench +2.21 [+1.14, +3.28] is the lone CI-clean positive (hallucination axis transfer).

**Spec source-of-truth.** `docs/paper/emnlp_outline_ko.md` — §6.3 (Capability preservation).

This notebook drives heavy inference stages by `subprocess`-invoking the
existing drivers in `scripts/`. The `RUN_INFERENCE = False` toggle below
lets a reviewer read the full pipeline without GPU access — canonical
CSVs are read straight from disk and figures get re-rendered from them.


## 1 · Setup — paths + subprocess helper

In [ ]:
from __future__ import annotations
import json
import os
import shlex
import subprocess
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_main_worktree() -> Path:
    common = subprocess.check_output(
        ["git", "rev-parse", "--git-common-dir"], cwd=Path.cwd(), text=True
    ).strip()
    return Path(common).resolve().parent


def find_worktree_root() -> Path:
    return Path(subprocess.check_output(
        ["git", "rev-parse", "--show-toplevel"], cwd=Path.cwd(), text=True
    ).strip()).resolve()


MAIN     = find_main_worktree()
WORKTREE = find_worktree_root()

# Scripts + configs from the active branch (worktree); gitignored artifacts
# (inputs/, outputs/, docs/insights/_data/) from MAIN.
SCRIPTS    = WORKTREE / "scripts"
CONFIGS    = WORKTREE / "configs"
DATA_DIR   = MAIN / "docs" / "insights" / "_data"
FIGURES    = WORKTREE / "docs" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

GPUS = os.environ.get("VLM_ANCHOR_GPUS", "0,1,2,3,4")
RUN_INFERENCE = False  # set True to invoke the heavy driver(s)

print(f"MAIN     = {MAIN}")
print(f"WORKTREE = {WORKTREE}")
print(f"GPUS     = {GPUS}")
print(f"RUN_INFERENCE = {RUN_INFERENCE}")


In [ ]:
def run_cmd(cmd: list[str] | str, *, dry: bool = False, env: dict | None = None) -> int:
    printable = " ".join(shlex.quote(c) for c in cmd) if isinstance(cmd, list) else cmd
    print(f"$ {printable}")
    if dry:
        print("  (dry — RUN_INFERENCE=False)")
        return 0
    full_env = os.environ.copy()
    if env:
        full_env.update(env)
    return subprocess.run(cmd, cwd=MAIN, env=full_env,
                          shell=isinstance(cmd, str)).returncode


def save_figure(fig, stem: str, png_dir: Path = FIGURES,
                pdf_dir: Path | None = None):
    png = png_dir / f"{stem}.png"
    fig.savefig(png, bbox_inches="tight", dpi=160)
    print(f"wrote {png}")
    if pdf_dir is not None:
        pdf_dir.mkdir(parents=True, exist_ok=True)
        pdf = pdf_dir / f"{stem}.pdf"
        fig.savefig(pdf, bbox_inches="tight")
        print(f"wrote {pdf}")


## 2 · Capability eval — 6 benchmarks via VLMEvalKit

`scripts/run_capability_eval.py` is a per-benchmark interleaving driver
that runs baseline (vanilla `LLaVA-OneVision-HF`) and `+mit`
(`LLaVAOneVisionMitigated` wrapper that hooks the K=8 subspace projection
at L=26) on each benchmark sequentially. Live progress lands in
`outputs/capability_eval/progress.log` and a partial CSV is refreshed
after every benchmark.

**Wall time.** ~13 h on 1 GPU for the full 6-benchmark sweep
(POPE n=5127 is the long pole; HallusionBench n=951 is the first
CI-clean signal at ~2 h). Smoke-mode (`--max-questions 50`) finishes
in ~30 min.


In [ ]:
def run_capability_eval(config: str = "capability_eval.yaml",
                         max_questions: int | None = None):
    cmd = [
        "uv", "run", "python", str(SCRIPTS / "run_capability_eval.py"),
        "--config", str(CONFIGS / config),
    ]
    if max_questions is not None:
        cmd += ["--max-questions", str(max_questions)]
    return run_cmd(cmd, dry=not RUN_INFERENCE)


# Default: full 6-benchmark eval (~13 h). Toggle smoke=True for a
# 50-question subset (~30 min) to validate the pipeline before commit.
smoke = False
run_capability_eval(max_questions=50 if smoke else None)


## 3 · Aggregate partial → final


In [ ]:
def aggregate_capability():
    partial_csv = MAIN / "docs" / "insights" / "_data" / "capability_eval_partial.csv"
    partial_md  = MAIN / "docs" / "insights" / "_data" / "capability_eval_partial.md"
    final_csv   = DATA_DIR / "capability_eval_per_benchmark.csv"
    final_md    = DATA_DIR / "capability_eval_per_benchmark.md"
    cmd = [
        "uv", "run", "python", str(SCRIPTS / "aggregate_capability_eval.py"),
        "finalize",
        "--partial-csv", str(partial_csv),
        "--partial-md",  str(partial_md),
        "--final-csv",   str(final_csv),
        "--final-md",    str(final_md),
    ]
    return run_cmd(cmd, dry=not RUN_INFERENCE)


aggregate_capability()


## 4 · Table 6.3 — Per-benchmark capability deltas


In [ ]:
cap_csv = DATA_DIR / "capability_eval_per_benchmark.csv"
if cap_csv.exists():
    df = pd.read_csv(cap_csv)
    # Render in the same column order as paper Table 6.3.
    df = df.assign(
        baseline=df["acc_baseline"] * 100,
        mit=df["acc_mit"] * 100,
        delta_pp=df["delta"] * 100,
        ci_low_pp=df["ci_low"] * 100,
        ci_high_pp=df["ci_high"] * 100,
    )
    rows = []
    for _, r in df.iterrows():
        rows.append({
            "Benchmark": r["benchmark"],
            "n":         int(r["n"]),
            "baseline":  f"{r['baseline']:.2f}",
            "+mit":      f"{r['mit']:.2f}",
            "Δ (pp)":    f"{r['delta_pp']:+.2f}",
            "95% CI":    f"[{r['ci_low_pp']:+.2f}, {r['ci_high_pp']:+.2f}]",
            "status":    r["status"],
        })
    macro_delta = df["delta"].mean() * 100
    print(pd.DataFrame(rows).to_string(index=False))
    print()
    print(f"Macro Δ  = {macro_delta:+.2f} pp   (STRICT_FREE_LUNCH protocol)")
else:
    print(f"missing: {cap_csv}")


## 5 · Figure §6.3 — capability bar chart (optional)

A bar chart of per-benchmark Δ with 95 % CI whiskers — paper publishes
Table 6.3 only, so this figure is for the appendix / supplementary slot.


In [ ]:
def fig_capability() -> plt.Figure | None:
    src = DATA_DIR / "capability_eval_per_benchmark.csv"
    if not src.exists():
        print(f"  (skipped — {src} missing)")
        return None
    df = pd.read_csv(src)
    # Order by absolute Δ descending so signal benchmarks lead.
    df = df.assign(absd=df["delta"].abs()).sort_values("absd", ascending=False)
    fig, ax = plt.subplots(figsize=(8.5, 4.2), dpi=150)
    y = np.arange(len(df))
    deltas = df["delta"].values * 100
    yerr_lo = deltas - df["ci_low"].values * 100
    yerr_hi = df["ci_high"].values * 100 - deltas
    colors = ["#2ca25f" if (lo > 0 and hi > 0) else "#9e9e9e"
              for lo, hi in zip(df["ci_low"]*100, df["ci_high"]*100)]
    ax.barh(y, deltas, color=colors, alpha=0.7, edgecolor="black", linewidth=0.4)
    ax.errorbar(deltas, y, xerr=[yerr_lo, yerr_hi], fmt="none",
                ecolor="black", linewidth=0.8, capsize=3)
    ax.axvline(0, color="#888", linewidth=0.7)
    ax.set_yticks(y)
    ax.set_yticklabels(df["benchmark"].values)
    ax.set_xlabel("Δ accuracy (pp)  — mitigation vs baseline")
    ax.set_title("§6.3 — Capability preservation across 6 VLMEvalKit benchmarks\n"
                 "STRICT_FREE_LUNCH; green = 95 % CI excludes 0 (positive)")
    ax.grid(axis="x", linestyle=":", alpha=0.4)
    fig.tight_layout()
    return fig


fig = fig_capability()
if fig is not None:
    save_figure(fig, "paper_6_3_capability_eval")
fig


## Summary

Pipeline:
1. `run_capability_eval.py` — per-benchmark baseline + `+mit` runs via
   VLMEvalKit programmatic API. ~13 h wall on 1 GPU for the full 6-bench
   set; live partial CSV at every checkpoint.
2. `aggregate_capability_eval.py finalize` →
   `docs/insights/_data/capability_eval_per_benchmark.{csv,md}`.
3. Table 6.3 rendered from the canonical CSV.

Headline (paper §6.3):
- Macro Δ = **+0.41 pp** across 6 benchmarks (n_total = 10,507).
- HallusionBench **+2.21 pp [+1.14, +3.28]** — only CI-clean positive
  (hallucination-axis transfer; mitigation incidentally improves
  visual-distraction handling).
- 5 other benchmarks within ±1 pp band (POPE −0.06, MMStar +0.13,
  RealWorldQA +1.31, MMBench-DEV-EN −0.34, OCRBench −0.80).

Verdict: STRICT_FREE_LUNCH — no benchmark regresses with a CI-clean
negative. Mitigation preserves general capability while reducing
anchoring (§6.2).
